# 主動型投資- 資料處理
**以「週」為單位做台股的買賣，賺價差**
流程 (網路爬蟲都使用 `WeekTermInvestment_DataScrape` 的工具)：
1. 輸入台股代碼 (ABBR)
2. 檢查 ABBR 資料夾內的 5 份 CSV (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)
3. 沒有資料夾或檔案有誤時：先用 `WeekTermInvestment_DataScrape` 抓取新資料，抓取成功後再刪除 ABBR 資料夾內的舊資料 (若有的話)，再存入新資料
4. 設定分析用的週次變數

投資操作：
* 買_時機點：大戶籌碼比例增加 & 大戶借券量下降 & 融資(韭菜)餘額持平/下降，買進價格<=大戶平均成本
* 賣_時機點：大戶籌碼比例下降 & 大戶借券量增加 & 融資(韭菜)餘額持平/上升 & 融資維持率持平/下降，賣出價格>=韭菜平均成本

### 匯入Package

In [1]:
import ast
import builtins
import json
import re
import types
from datetime import datetime
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()  # 本 notebook 所在的資料夾，各股票的資料夾 (ABBR) 都放在這裡
DATASCRAPE_NOTEBOOK = BASE_DIR / "WeekTermInvestment_DataScrape.ipynb"


def _is_definition(node, namespace):
    """只保留 import、函式/類別定義、常數設定。
    常數設定的等號右邊不能呼叫函式 (以免執行到 input() 或爬蟲)，用到的變數也必須已經定義
    (例如執行格中的 outputs = [(df_inst, ...)] 會被略過，因為 df_inst 是爬蟲的結果)"""
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return True
    if isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
        sub_nodes = list(ast.walk(node.value))
        names = [n.id for n in sub_nodes if isinstance(n, ast.Name)]
        return not any(isinstance(n, ast.Call) for n in sub_nodes) and all(
            name in namespace or hasattr(builtins, name) for name in names
        )
    return False


def load_datascrape_tools(nb_path):
    """把 WeekTermInvestment_DataScrape.ipynb 當成 package 載入：
    只執行裡面的 import 與函式/常數定義，不會執行「輸入代碼 → 爬資料」那一格"""
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    module = types.ModuleType(Path(nb_path).stem)
    module.__file__ = str(nb_path)
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:  # 例如含有 %magic 的格子，不是工具定義，略過
            continue
        # 逐行判斷、逐行執行，才知道某個變數在這一行之前是否已經定義
        for node in tree.body:
            if _is_definition(node, module.__dict__):
                exec(compile(ast.Module(body=[node], type_ignores=[]), str(nb_path), "exec"), module.__dict__)
    return module


# 所有網路爬蟲 (籌碼資料、週K線股價) 都在 DataScrape 中
ds = load_datascrape_tools(DATASCRAPE_NOTEBOOK)
print(f"已載入 {DATASCRAPE_NOTEBOOK.name} 的工具")

已載入 WeekTermInvestment_DataScrape.ipynb 的工具


### 函式定義(週資料檔案檢查)

In [ ]:
# ===== 週資料：檢查 ABBR 資料夾內的 5 份 CSV，沒有或有誤就用 DataScrape 重新取得 =====
# 5 份資料的檔名標籤與欄位定義在 DataScrape 的 WEEK_FILE_COLUMNS
# (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)


def _is_yyyymmdd(s):
    try:
        datetime.strptime(s, "%Y%m%d")
        return True
    except ValueError:
        return False


def _latest_week_file(folder, abbr, file_tag):
    """回傳資料夾內 ABBR_{file_tag}_YYYYMMDD.csv 中日期最新的那份，沒有就回傳 None"""
    pattern = re.compile(rf"{abbr}_{file_tag}_(\d{{8}})\.csv")
    matches = [f for f in Path(folder).glob("*.csv") if (m := pattern.fullmatch(f.name)) and _is_yyyymmdd(m.group(1))]
    return max(matches, key=lambda f: f.name, default=None)


def check_week_files(folder, abbr):
    """檢查資料夾內的 5 份週資料 CSV，回傳問題清單 (空清單代表檔案都正確)
    檢查項目：至少 5 份 CSV、5 種檔名都有 (結尾是合法的年月日)、檔案讀得到且欄位數正確、有資料"""
    n_csv = len(list(Path(folder).glob("*.csv")))
    problems = []
    if n_csv < len(ds.WEEK_FILE_COLUMNS):
        problems.append(f"CSV 檔只有 {n_csv} 份，至少要 {len(ds.WEEK_FILE_COLUMNS)} 份")
    for tag, columns in ds.WEEK_FILE_COLUMNS.items():
        latest = _latest_week_file(folder, abbr, tag)  # 同一種資料有多份時，檢查日期最新的那份
        if latest is None:
            problems.append(f"缺少 {abbr}_{tag}_檔案建立年月日.csv")
            continue
        try:
            df = pd.read_csv(latest, encoding="utf-8-sig")
        except Exception as e:
            problems.append(f"{latest.name} 無法讀取：{e}")
            continue
        if df.shape[1] != len(columns):
            problems.append(f"{latest.name} 有 {df.shape[1]} 欄，應該是 {len(columns)} 欄")
        elif df.empty:
            problems.append(f"{latest.name} 沒有資料")
    return problems


def ensure_week_data(abbr):
    """確保 ABBR 資料夾內有正確的 5 份週資料 CSV，回傳資料夾路徑
    沒有資料夾或檔案有誤時：先用 DataScrape 抓取 5 份新資料，全部成功後才刪除資料夾內舊的 CSV (若有的話)，
    再存入新資料；抓取失敗時舊檔都不會被刪除"""
    folder = BASE_DIR / abbr
    if not folder.is_dir():
        print(f"⚠ 找不到資料夾「{abbr}」")
    else:
        problems = check_week_files(folder, abbr)
        if not problems:
            print(f"✔ 資料夾「{abbr}」內的 {len(ds.WEEK_FILE_COLUMNS)} 份 CSV 檔都正確")
            return folder
        print(f"⚠ 資料夾「{abbr}」內的檔案有誤：")
        for p in problems:
            print(f"   - {p}")

    print("使用 WeekTermInvestment_DataScrape 抓取新資料，抓取成功後再刪除舊的 CSV 檔…")
    week_data = ds.fetch_all_week_data(abbr)
    for old in sorted(folder.glob("*.csv")):  # 資料夾不存在時沒有舊檔
        old.unlink()
        print(f"已刪除舊檔：{old.name}")
    for file_tag, df in week_data.items():
        csv_path = ds.save_week_csv(df, abbr, file_tag, base_dir=BASE_DIR)
        print(f"{df.attrs['symbol']} {file_tag} 共 {len(df)} 筆，已存檔：{csv_path}")

    problems = check_week_files(folder, abbr)
    if problems:
        raise RuntimeError(f"重新取得資料後，資料夾「{abbr}」內的檔案仍有誤：\n" + "\n".join(problems))
    print(f"✔ 資料夾「{abbr}」內的 {len(ds.WEEK_FILE_COLUMNS)} 份 CSV 檔都正確")
    return folder

### 函式定義(大戶買入平均成本)
以 `PIVOT_YEAR_WEEK` 為錨點(包含)，往前 `INPUT_WEEK_NUM` 筆法人買賣超週資料，外資、投信+自營商 各自計算期間內的買進均價：
* 賣超不影響成本，只用買超的週：平均成本 = Σ(買超張數 × 當週典型價) / Σ(買超張數)；期間內都沒有買超時為 NaN
* 當週典型價 = (高 + 低 + 收) / 3，來自 `ABBR_Stock_Price_Week`，與法人資料用 ISO 年週對齊
* 投信、自營商視為同一個大戶，同一週的買賣超先相加，再判斷是買超或賣超
* 期初持股與成本無法得知，所以這是大戶「期間內新買進部位」的平均成本

In [ ]:
# ===== 平均成本：共用函式 + 大戶（外資、投信+自營商）買入平均成本 =====
# 週次一律用 ISO 年週 (年, 第幾週)：週一到週日為一週，與 date.isocalendar() 相同，
# 例如 date(2026, 11, 6).isocalendar()[:2] == (2026, 45)
# 各份 CSV 同一週的日期不一定相同 (例如法人用週五 11/6、資券餘額用週一 11/2)，所以用 ISO 年週對齊

# 大戶分組: 法人買賣超 CSV 中要加總的欄位 (投信+自營商 視為同一個大戶，同一週先相加)
INSTITUTION_GROUPS = {"外資": ["外資(張)"], "投信+自營商": ["投信(張)", "自營商(張)"]}
INSTITUTIONAL_FILE_TAG = "Institutional_Trading_Week_8cols"


def _to_number(col):
    """CSV 中的數字可能是有千分位逗號的字串 (例如 -2,927)，轉成數值；無法轉換的變成 NaN"""
    return pd.to_numeric(col.astype(str).str.replace(",", ""), errors="coerce")


def read_week_csv(abbr, file_tag):
    """讀取 ABBR 資料夾中日期最新的 ABBR_{file_tag}_YYYYMMDD.csv，
    依日期由新到舊排序，並加上「年週」欄位 (ISO 年, 第幾週)"""
    path = _latest_week_file(BASE_DIR / abbr, abbr, file_tag)
    if path is None:
        raise FileNotFoundError(f"資料夾「{abbr}」中找不到 {abbr}_{file_tag}_檔案建立年月日.csv")
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["_date"] = pd.to_datetime(df.iloc[:, 0], format="%Y/%m/%d")  # 第一欄是日期
    df = df.sort_values("_date", ascending=False).reset_index(drop=True)
    df["年週"] = [d.isocalendar()[:2] for d in df.pop("_date")]
    return df


def _week_window(df, pivot_year_week, input_week_num, label):
    """從 read_week_csv 讀到的週資料 (由新到舊) 中，取出以 pivot_year_week 為終點 (包含) 往前 input_week_num 筆"""
    pivot_year_week = tuple(pivot_year_week)
    positions = [i for i, yw in enumerate(df["年週"]) if yw == pivot_year_week]
    if not positions:
        raise ValueError(
            f"{label}資料中沒有 {pivot_year_week} 這一週 (資料範圍 {df['年週'].iloc[-1]} ~ {df['年週'].iloc[0]})"
        )
    window = df.iloc[positions[0] : positions[0] + input_week_num]
    if len(window) < input_week_num:
        raise ValueError(f"{pivot_year_week} 往前只有 {len(window)} 筆{label}資料，不足 {input_week_num} 筆")
    return window


def _week_typical_price(abbr, window):
    """回傳 window 中每一週的典型價 (高 + 低 + 收) / 3 (比只用收盤價更接近整週的成交價位)，index 與 window 相同"""
    price = read_week_csv(abbr, ds.PRICE_FILE_TAG)
    typical = (_to_number(price["高"]) + _to_number(price["低"]) + _to_number(price["收"])) / 3
    price_by_week = dict(zip(price["年週"], typical))
    missing = [yw for yw in window["年週"] if pd.isna(price_by_week.get(yw))]
    if missing:
        raise ValueError(f"股價資料中沒有這些週的高/低/收：{missing}")
    return pd.Series([price_by_week[yw] for yw in window["年週"]], index=window.index)


def _buy_avg_cost(window, lots, week_price, label):
    """賣出不影響成本，只用買進 (張數 > 0) 的週：平均成本 = Σ(買進張數 × 當週典型價) / Σ(買進張數)
    期初持股與成本無法得知，所以這是「期間內新買進部位」的平均成本；期間內都沒有買進時是 NaN"""
    if lots.isna().any():
        raise ValueError(f"{label} 的張數有無法轉成數字的值，年週：{window.loc[lots.isna(), '年週'].tolist()}")
    buys = lots > 0
    total_lots = lots[buys].sum()
    total_money = (lots[buys] * week_price[buys]).sum()  # 張 × 元；1 張 = 1000 股，相除時單位抵銷，即每股價格
    return total_money / total_lots if total_lots > 0 else float("nan")


def calc_institutional_avg_cost(abbr, pivot_year_week, input_week_num):
    """計算大戶 (外資、投信+自營商) 在期間內的買進均價，回傳 {"外資": 平均成本, "投信+自營商": 平均成本}
    - 以 pivot_year_week (ISO 年, 第幾週) 為終點 (包含)，往前取 input_week_num 筆法人買賣超週資料
    - 只用買超的週：平均成本 = Σ(買超張數 × 當週典型價) / Σ(買超張數)；期間內都沒有買超時是 NaN"""
    inst = read_week_csv(abbr, INSTITUTIONAL_FILE_TAG)
    window = _week_window(inst, pivot_year_week, input_week_num, "法人買賣超")
    week_price = _week_typical_price(abbr, window)
    return {
        name: _buy_avg_cost(window, sum(_to_number(window[c]) for c in columns), week_price, name)
        for name, columns in INSTITUTION_GROUPS.items()
    }

### 函式定義(散戶買入平均成本)
以 `PIVOT_YEAR_WEEK` 為錨點(包含)，往前 `INPUT_WEEK_NUM` 筆資券餘額週資料，算法比照大戶：
* 融資減少不影響成本，只用融資增加的週：平均成本 = Σ(融資增加張數 × 當週典型價) / Σ(融資增加張數)；期間內融資都沒有增加時為 NaN
* 融資增減取自 `ABBR_Margin_Week_11cols` 的「融資 → 增減」；當週典型價 = (高 + 低 + 收) / 3，與資券資料用 ISO 年週對齊
* 期初融資部位與成本無法得知，所以這是散戶「期間內新增融資部位」的平均成本

In [ ]:
# ===== 散戶（融資）買入平均成本 =====

MARGIN_FILE_TAG = "Margin_Week_11cols"
# 資券餘額 CSV 有兩個「增減」欄：第一個是融資、第二個是融券 (pandas 讀進來會改名為「增減.1」)
MARGIN_CHANGE_COLUMN = "增減"


def calc_retail_avg_cost(abbr, pivot_year_week, input_week_num):
    """計算散戶 (融資) 在期間內的買進均價，回傳平均成本
    - 以 pivot_year_week (ISO 年, 第幾週) 為終點 (包含)，往前取 input_week_num 筆資券餘額週資料
    - 融資減少 (賣出還款) 不影響成本，只用融資增加的週：
      平均成本 = Σ(融資增加張數 × 當週典型價) / Σ(融資增加張數)；期間內融資都沒有增加時是 NaN"""
    margin = read_week_csv(abbr, MARGIN_FILE_TAG)
    window = _week_window(margin, pivot_year_week, input_week_num, "資券餘額")
    lots = _to_number(window[MARGIN_CHANGE_COLUMN])  # 每週融資增減張數
    return _buy_avg_cost(window, lots, _week_typical_price(abbr, window), "融資")

### 輸入公司代碼，檢查週資料(5份csv)

In [5]:
# ===== 執行：輸入代碼 → 檢查 ABBR 資料夾的 5 份 CSV (沒有或有誤就用 DataScrape 重新取得) =====
# 重新取得時想看瀏覽器操作過程，可改用 ds.fetch_all_week_data(ABBR, headless=False)
ABBR = ds.ask_stock_code()
ABBR_DIR = ensure_week_data(ABBR)

請輸入台股代碼(4位數字)：2330
✔ 資料夾「2330」內的 5 份 CSV 檔都正確


### 分析參數
* `PIVOT_YEAR_WEEK`：選定的年與該年的第幾週，型別為 `(年, 第幾週)` 的 tuple，採 ISO 週次 (週一到週日為一週)，例如 `(2026, 45)` 是 2026/11/2 ~ 11/8 那一週；某天屬於第幾週可用 `date(2026, 11, 6).isocalendar()[:2]` 查
* `INPUT_WEEK_NUM`：以 `PIVOT_YEAR_WEEK` 為終點(包含)，往前幾週的數據做數值計算
* `OUTPUT_WEEK_NUM`：以 `PIVOT_YEAR_WEEK` 為起點(不包含)，往後幾週的數據做驗證

In [ ]:
PIVOT_YEAR_WEEK = (2026, 45)  # 選定的年與該年的第幾週 (ISO 年, 第幾週)
INPUT_WEEK_NUM = 12           # 以 PIVOT_YEAR_WEEK 為終點(包含)，往前幾週的數據做計算
OUTPUT_WEEK_NUM = None        # 以 PIVOT_YEAR_WEEK 為起點(不包含)，往後幾週的數據做驗證

### 大戶買入平均成本

In [ ]:
# ===== 執行：計算大戶買入平均成本 =====
avg_cost = calc_institutional_avg_cost(ABBR, PIVOT_YEAR_WEEK, INPUT_WEEK_NUM)
for name, cost in avg_cost.items():
    print(f"{ABBR} {name} 平均成本 ({PIVOT_YEAR_WEEK} 往前 {INPUT_WEEK_NUM} 週)：{cost:.2f}")

### 散戶買入平均成本

In [ ]:
# ===== 執行：計算散戶買入平均成本 =====
retail_cost = calc_retail_avg_cost(ABBR, PIVOT_YEAR_WEEK, INPUT_WEEK_NUM)
print(f"{ABBR} 散戶(融資) 平均成本 ({PIVOT_YEAR_WEEK} 往前 {INPUT_WEEK_NUM} 週)：{retail_cost:.2f}")